# Imports

In [1]:
import glob
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# Fixed s = 1287

## Get data

In [2]:
data_1287 = pd.read_csv("results/result_1287.csv", index_col=0)

In [3]:
data_1287.head()

,method,s,N,k,iter,error,diff_max,time
0,DI,1287,1999,1,1,1.580182e-13,5.773160e-15,0.087045
1,WMI,1287,1999,1,1,1.583327e-13,6.661338e-15,0.011337
2,ISM,1287,1999,1,1,1.582999e-13,6.661338e-15,0.011218
3,DI,1287,1999,1,2,1.580182e-13,5.773160e-15,0.085403
4,WMI,1287,1999,1,2,1.583327e-13,6.661338e-15,0.011235


In [4]:
data_1287_grouped = data_1287.groupby(["k", "method"]).agg({"iter": "max", "error": ["mean", "std"], "diff_max": "mean", "time": ["mean", "std"]})
if isinstance(data_1287_grouped.columns, pd.MultiIndex):
    data_1287_grouped.columns = ['_'.join(col).strip() for col in data_1287_grouped.columns.values]
data_1287_grouped.head()

iter_max    error_mean  error_std  diff_max_mean  time_mean  \
k method                                                                
1 DI           200  1.580182e-13        0.0   5.773160e-15   0.085946   
  ISM          200  1.582999e-13        0.0   6.661338e-15   0.007743   
  WMI          200  1.583327e-13        0.0   6.661338e-15   0.007787   
2 DI           200  1.593879e-13        0.0   6.439294e-15   0.084217   
  ISM          200  2.902351e+00        0.0   2.980472e-02   0.011417   

          time_std  
k method            
1 DI      0.003124  
  ISM     0.001375  
  WMI     0.001376  
2 DI      0.002697  
  ISM     0.000356

In [5]:
data_1287_grouped_for_plot = data_1287_grouped.reset_index()

## Plot number of new data vs. execution time for each method

In [6]:
fig_k_time = px.line(data_1287_grouped_for_plot, x="k", y="time_mean", color="method", error_y="time_std", markers=True, color_discrete_sequence=["red", "green", "blue"])
fig_k_time.update_layout(
    title="Average execution time for each method based on number of new data (k)",
    xaxis_title="k (Number of new data)",
    yaxis_title="Average Execution Time",
    yaxis_type="log",
    xaxis_type="log",
    legend_title="Method",
    template="simple_white",
    font=dict(size=12),
    width=1000,
    height=500
)
fig_k_time.show()

## Plot number of new data vs. norm error for each method

In [7]:
fig_k_error = px.line(data_1287_grouped_for_plot, x="k", y="error_mean", color="method", error_y="error_std", markers=True, color_discrete_sequence=["red", "green", "blue"])
fig_k_error.update_layout(
    title="Average error for each method based on number of new data (k)",
    xaxis_title="k (Number of new data)",
    yaxis_title="Average Error",
    yaxis_type="log",
    xaxis_type="log",
    legend_title="Method",
    template="simple_white",
    font=dict(size=12),
    width=1000,
    height=500
)
fig_k_error.show()

## Convert results in latex

In [8]:
def create_latex_table(data, x_axis_col="k", y_axis_col="method", value_col="time_mean", y_axis_order=["DI", "ISM", "WMI"]):
    """
    Create a LaTeX table with value_col grouped by x_axis_col and y_axis_col.
    
    Parameters:
    data: DataFrame with columns k, method, and time_mean
    x_axis_col: column name for the values on each line
    y_axis_col: column name for the values on each column
    time_col: column name for the values that fill the table
    
    Returns:
    str: LaTeX table string
    """
    if y_axis_order is not None:
        # Reorder the columns of the table in the data
        data = data.copy()
        data[y_axis_col] = pd.Categorical(data[y_axis_col], categories=y_axis_order, ordered=True)
        data = data.sort_values(y_axis_col)

    # Pivot the data to have x_axis_col as rows and y_axis_col as columns
    pivot_table = data.pivot(index=x_axis_col, columns=y_axis_col, values=value_col)
    
    # Get the column names
    column_names = pivot_table.columns.tolist()
    
    # Create LaTeX table header
    latex_table = "\\begin{tabular}{|l|" + "|".join(["c"] * len(column_names)) + "|}\n"
    latex_table += "\\hline\n"
    
    # Add header row
    header_row = "\\textbf{k}"
    for column in column_names:
        header_row += f" & \\textbf{{{column}}}"
    latex_table += header_row + " \\\\\\hline\n"
    
    previous_sorted_indices = None
    # Add data rows
    for x_value in pivot_table.index:
        row_data = []
        x_str = str(x_value)
        
        # Get the value for this x_axis value
        values = pivot_table.loc[x_value].values
        
        # Find sorted indices (ascending order)
        sorted_indices = np.argsort(values)
        
        # Identify best and second best
        best_idx = sorted_indices[0]
        second_best_idx = sorted_indices[1] if len(sorted_indices) > 1 else None
        
        # Add each value with appropriate formatting
        for i, (column, val) in enumerate(zip(column_names, values)):
            if i == best_idx:
                formatted_val = f"\\textbf{{{val:.7g}}}"
            elif i == second_best_idx:
                formatted_val = f"\\underline{{\\textit{{{val:.7g}}}}}"
            else:
                formatted_val = f"{val:.7g}"
            row_data.append(formatted_val)
        
        # If there is a change in the order of the methods, add an \hline
        if previous_sorted_indices is not None and any(previous_sorted_indices != sorted_indices):
            latex_table += "\\hline\n"
        
        previous_sorted_indices = sorted_indices
        latex_table += f"{x_str} & {' & '.join(row_data)} \\\\\\hline\n"
    
    latex_table += "\\end{tabular}"
    
    return latex_table

In [9]:
print(create_latex_table(data_1287_grouped_for_plot))

\begin{tabular}{|l|c|c|c|}
\hline
\textbf{k} & \textbf{DI} & \textbf{ISM} & \textbf{WMI} \\\hline
1 & 0.08594638 & \textbf{0.007743145} & \underline{\textit{0.0077865}} \\\hline
\hline
2 & 0.08421694 & \underline{\textit{0.01141714}} & \textbf{0.007551792} \\\hline
3 & 0.08086838 & \underline{\textit{0.01424055}} & \textbf{0.005989846} \\\hline
4 & 0.07941904 & \underline{\textit{0.02148158}} & \textbf{0.005903258} \\\hline
5 & 0.07437832 & \underline{\textit{0.01883538}} & \textbf{0.004278867} \\\hline
10 & 0.08314858 & \underline{\textit{0.06215603}} & \textbf{0.008458068} \\\hline
\hline
20 & \underline{\textit{0.08020028}} & 0.09959804 & \textbf{0.008605508} \\\hline
30 & \underline{\textit{0.08923532}} & 0.1900465 & \textbf{0.01317246} \\\hline
40 & \underline{\textit{0.08028041}} & 0.1966603 & \textbf{0.01141213} \\\hline
50 & \underline{\textit{0.08450316}} & 0.2969778 & \textbf{0.01532174} \\\hline
100 & \underline{\textit{0.08219219}} & 0.4851311 & \textbf{0.02341882} \\\hline

In [10]:
print(create_latex_table(data_1287_grouped_for_plot, value_col="error_mean"))

\begin{tabular}{|l|c|c|c|}
\hline
\textbf{k} & \textbf{DI} & \textbf{ISM} & \textbf{WMI} \\\hline
1 & \textbf{1.580182e-13} & \underline{\textit{1.582999e-13}} & 1.583327e-13 \\\hline
\hline
2 & \textbf{1.593879e-13} & 2.902351 & \underline{\textit{1.605027e-13}} \\\hline
3 & \textbf{1.594071e-13} & 4.358941 & \underline{\textit{1.595547e-13}} \\\hline
4 & \textbf{1.593055e-13} & 5.333518 & \underline{\textit{1.609318e-13}} \\\hline
5 & \textbf{1.59972e-13} & 6.150092 & \underline{\textit{1.604706e-13}} \\\hline
10 & \textbf{1.598138e-13} & 9.544279 & \underline{\textit{1.610103e-13}} \\\hline
20 & \textbf{1.606954e-13} & 14.10532 & \underline{\textit{1.628296e-13}} \\\hline
30 & \textbf{1.607787e-13} & 17.76659 & \underline{\textit{1.641526e-13}} \\\hline
40 & \textbf{1.607618e-13} & 20.82128 & \underline{\textit{1.662739e-13}} \\\hline
50 & \textbf{1.61661e-13} & 23.75783 & \underline{\textit{1.677276e-13}} \\\hline
100 & \textbf{1.618503e-13} & 38.34813 & \underline{\textit{1.773381

# s ranging in [10, 20, 50, 100, 250, 500, 750, 1000, 1287]

## Get data

In [11]:
result_files = glob.glob("results/result_*.csv")

result_list = []
for file_name in result_files:
    result_list.append(pd.read_csv(file_name, index_col=0))
    
data_s = pd.concat(result_list)

In [12]:
data_s.head()

,method,s,N,k,iter,error,diff_max,time
0,DI,10,1999,1,1,8.892186e-16,5.551115e-16,0.000074
1,WMI,10,1999,1,1,7.038792e-16,4.440892e-16,0.000098
2,ISM,10,1999,1,1,7.041612e-16,4.440892e-16,0.000048
3,DI,10,1999,1,2,8.892186e-16,5.551115e-16,0.000036
4,WMI,10,1999,1,2,7.038792e-16,4.440892e-16,0.000032


In [13]:
# Average time per method for each s and k
avg_times = data_s.groupby(["s", "k", "method"])["time"].mean().reset_index()

# Keep the minimum average time
min_avg_times = avg_times.groupby(["s", "k"])["time"].min().reset_index()
min_avg_times = min_avg_times.rename(columns={"time": 'min_avg_time'})

# Merged the min time into the average times
merged_data = avg_times.merge(min_avg_times, on=["s", "k"])
# Keep the method with the min time
final_data = avg_times[avg_times["time"] == merged_data['min_avg_time']]

## Plot fastest method depending on matrix size (s) and number of new data (k)

In [14]:
fig_fastest = px.scatter(
    final_data,
    x="s",
    y="k",
    color="method",
    color_discrete_sequence=["red", "green", "blue"],
    title="Fastest method by matrix size (s) and number of new data (k)",
    labels={
        "s": "s (Matrix size)",
        "k": "k (Number of new data)"
    }
)

if not final_data.empty:
    s_min = final_data["s"].min()
    s_max = final_data["s"].max()
    s_range = np.linspace(s_min, s_max, 100)
    k_line = s_range / 3 + 3
    
    fig_fastest.add_trace(go.Scatter(
        x=s_range,
        y=k_line,
        mode='lines',
        line=dict(dash='dash', color='black'),
        name='k = s/3 + 3',
        showlegend=True
    ))

fig_fastest.update_layout(
    legend_title="Method",
    yaxis_type="log",
    xaxis_type="log",
    template="simple_white",
    font=dict(size=12),
    width=800,
    height=600
)

fig_fastest.update_traces(
    marker=dict(size=10, opacity=0.8)
)

fig_fastest.write_html("images/fastest_method.html")

fig_fastest.show()